# Spotify Top 50 Tracks of 2020 Data Analysis

This notebook follows the assignment requirements in order. The goal is to load, clean, and analyze the Spotify Top 50 Tracks of 2020 dataset using Pandas.

EDA means Exploratory Data Analysis. In plain English, it means looking through the data carefully to understand patterns before making conclusions.

## 1. Download the Dataset

The dataset used for this project is the Spotify Top 50 Tracks of 2020 dataset from Kaggle. The CSV file has already been downloaded and placed in the project folder:

`tasks/Spotify Data Analysis/data/spotifytoptracks.csv`

## 2. Load the Data Using Pandas

Pandas is a Python library used for working with table-shaped data. I use `read_csv()` to load the CSV file into a DataFrame.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
data_path = Path("../data/spotifytoptracks.csv")

raw_df = pd.read_csv(data_path)
raw_df.head()

,Unnamed: 0,artist,album,track_name,track_id,energy,danceability,key,loudness,acousticness,speechiness,instrumentalness,liveness,valence,tempo,duration_ms,genre
0,0,The Weeknd,After Hours,Blinding Lights,0VjIjW4GlUZAMYd2vXMi3b,0.730,0.514,1,-5.934,0.00146,0.0598,0.000095,0.0897,0.334,171.005,200040,R&B/Soul
1,1,Tones And I,Dance Monkey,Dance Monkey,1rgnBhdG2JDFTbYkYRZAku,0.593,0.825,6,-6.401,0.68800,0.0988,0.000161,0.1700,0.540,98.078,209755,Alternative/Indie
2,2,Roddy Ricch,Please Excuse Me For Being Antisocial,The Box,0nbXyq5TXYPCO7pr3N8S4I,0.586,0.896,10,-6.687,0.10400,0.0559,0.000000,0.7900,0.642,116.971,196653,Hip-Hop/Rap
3,3,SAINt JHN,Roses (Imanbek Remix),Roses - Imanbek Remix,2Wo6QQD1KMDWeFkkjLqwx5,0.721,0.785,8,-5.457,0.01490,0.0506,0.004320,0.2850,0.894,121.962,176219,Dance/Electronic
4,4,Dua Lipa,Future Nostalgia,Don't Start Now,3PfIrDoz19wz7qK7tYeu62,0.793,0.793,11,-4.521,0.01230,0.0830,0.000000,0.0951,0.679,123.950,183290,Nu-disco


The dataset loaded successfully. Each row represents one track, and each column gives information about that track.

## 3. Data Cleaning

Before answering the analysis questions, I clean the dataset by checking missing values, duplicate samples, duplicate features, unnecessary columns, and possible outliers.

### 3.1 Handle Missing Values

Missing values are empty cells. I check each column to see whether any values are missing.

In [3]:
missing_values = raw_df.isnull().sum()
missing_values

Unnamed: 0          0
artist              0
album               0
track_name          0
track_id            0
energy              0
danceability        0
key                 0
loudness            0
acousticness        0
speechiness         0
instrumentalness    0
liveness            0
valence             0
tempo               0
duration_ms         0
genre               0
dtype: int64

All columns have 0 missing values, so no rows or columns need to be changed for missing data.

### 3.2 Remove Duplicate Samples

Duplicate samples are repeated rows. I check whether the exact same row appears more than once.

In [4]:
duplicate_sample_count = raw_df.duplicated().sum()
duplicate_sample_count

np.int64(0)

There are 0 duplicate samples, so no duplicate rows need to be removed.

### 3.3 Remove Duplicate Features

Duplicate features are repeated columns. I check for duplicate column names and columns with identical contents.

In [5]:
duplicate_feature_names = raw_df.columns[raw_df.columns.duplicated()].tolist()

columns = raw_df.columns.tolist()
duplicate_feature_contents = []

for first_index in range(len(columns)):
    for second_index in range(first_index + 1, len(columns)):
        first_column = columns[first_index]
        second_column = columns[second_index]
        if raw_df[first_column].equals(raw_df[second_column]):
            duplicate_feature_contents.append((first_column, second_column))

print("Duplicate feature names:", duplicate_feature_names)
print("Duplicate feature contents:", duplicate_feature_contents)

Duplicate feature names: []
Duplicate feature contents: []


There are no duplicate feature names and no columns with identical contents. This means no duplicate features need to be removed.

### 3.4 Remove Unnecessary Column

The column `Unnamed: 0` only repeats the row numbers from the CSV file. It is not a music feature, so I remove it before analysis.

In [6]:
raw_df["Unnamed: 0"].head()

0    0
1    1
2    2
3    3
4    4
Name: Unnamed: 0, dtype: int64

In [7]:
df = raw_df.drop(columns=["Unnamed: 0"]).copy()
df.head()

,artist,album,track_name,track_id,energy,danceability,key,loudness,acousticness,speechiness,instrumentalness,liveness,valence,tempo,duration_ms,genre
0,The Weeknd,After Hours,Blinding Lights,0VjIjW4GlUZAMYd2vXMi3b,0.730,0.514,1,-5.934,0.00146,0.0598,0.000095,0.0897,0.334,171.005,200040,R&B/Soul
1,Tones And I,Dance Monkey,Dance Monkey,1rgnBhdG2JDFTbYkYRZAku,0.593,0.825,6,-6.401,0.68800,0.0988,0.000161,0.1700,0.540,98.078,209755,Alternative/Indie
2,Roddy Ricch,Please Excuse Me For Being Antisocial,The Box,0nbXyq5TXYPCO7pr3N8S4I,0.586,0.896,10,-6.687,0.10400,0.0559,0.000000,0.7900,0.642,116.971,196653,Hip-Hop/Rap
3,SAINt JHN,Roses (Imanbek Remix),Roses - Imanbek Remix,2Wo6QQD1KMDWeFkkjLqwx5,0.721,0.785,8,-5.457,0.01490,0.0506,0.004320,0.2850,0.894,121.962,176219,Dance/Electronic
4,Dua Lipa,Future Nostalgia,Don't Start Now,3PfIrDoz19wz7qK7tYeu62,0.793,0.793,11,-4.521,0.01230,0.0830,0.000000,0.0951,0.679,123.950,183290,Nu-disco


The cleaned DataFrame is stored as `df`. Removing `Unnamed: 0` does not remove useful song information because Pandas already has row numbers on the left side of the table.

### 3.5 Treat Outliers

Outliers are values that are much higher or lower than most other values. I first review summary statistics, then use the IQR method to flag possible outliers.

In [8]:
df.describe()

,energy,danceability,key,loudness,acousticness,speechiness,instrumentalness,liveness,valence,tempo,duration_ms
count,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
mean,0.609300,0.716720,5.720000,-6.225900,0.256206,0.124158,0.015962,0.196552,0.555710,119.690460,199955.360000
std,0.154348,0.124975,3.709007,2.349744,0.265250,0.116836,0.094312,0.176610,0.216386,25.414778,33996.122488
min,0.225000,0.351000,0.000000,-14.454000,0.001460,0.029000,0.000000,0.057400,0.060500,75.801000,140526.000000
25%,0.494000,0.672500,2.000000,-7.552500,0.052800,0.048325,0.000000,0.093950,0.434000,99.557250,175845.500000
50%,0.597000,0.746000,6.500000,-5.991500,0.188500,0.070050,0.000000,0.111000,0.560000,116.969000,197853.500000
75%,0.729750,0.794500,8.750000,-4.285500,0.298750,0.155500,0.000020,0.271250,0.726250,132.317000,215064.000000
max,0.855000,0.935000,11.000000,-3.280000,0.934000,0.487000,0.657000,0.792000,0.925000,180.067000,312820.000000


In [9]:
df["duration_minutes"] = df["duration_ms"] / 1000 / 60

df[["track_name", "artist", "duration_ms", "duration_minutes"]].head()

,track_name,artist,duration_ms,duration_minutes
0,Blinding Lights,The Weeknd,200040,3.334000
1,Dance Monkey,Tones And I,209755,3.495917
2,The Box,Roddy Ricch,196653,3.277550
3,Roses - Imanbek Remix,SAINt JHN,176219,2.936983
4,Don't Start Now,Dua Lipa,183290,3.054833


In [10]:
outlier_columns = [
    "energy",
    "danceability",
    "loudness",
    "acousticness",
    "speechiness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_minutes",
]

outlier_summary = {}

for column in outlier_columns:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr
    outlier_count = df[
        (df[column] < lower_limit) | (df[column] > upper_limit)
    ].shape[0]
    outlier_summary[column] = outlier_count

pd.Series(outlier_summary, name="possible_outliers")

energy               0
danceability         3
loudness             1
acousticness         7
speechiness          6
instrumentalness    12
liveness             3
valence              0
tempo                0
duration_minutes     2
Name: possible_outliers, dtype: int64

Some possible outliers are flagged, but the values still make sense for songs. For example, a song can be unusually quiet, acoustic, or instrumental and still be valid. I do not remove these rows because they look like real music values, not data errors.

## 4. Exploratory Data Analysis

This section answers the assignment questions in the same order they were given.

### 4.1 How many observations are there in this dataset?

Observations are rows. In this dataset, each row is one track.

In [11]:
number_of_observations = df.shape[0]
number_of_observations

50

There are 50 observations, meaning the dataset contains 50 tracks.

### 4.2 How many features does this dataset have?

Features are columns. They describe each track.

In [12]:
number_of_features = df.shape[1]
number_of_features

17

The original dataset has 17 columns, including `Unnamed: 0`, which is an unnecessary index column. After removing it, there are 16 original useful features. I also created `duration_minutes` for easier interpretation, so the working DataFrame has 17 columns.

### 4.3 Which features are categorical?

Categorical features contain labels or groups, such as artist names and genres.

In [13]:
categorical_features = df.select_dtypes(include="object").columns.tolist()
categorical_features

['artist', 'album', 'track_name', 'track_id', 'genre']

The categorical features are text-based columns such as artist, album, track name, track ID, and genre.

### 4.4 Which features are numeric?

Numeric features contain numbers that can be measured or calculated.

In [14]:
numeric_features = df.select_dtypes(include="number").columns.tolist()
numeric_features

['energy',
 'danceability',
 'key',
 'loudness',
 'acousticness',
 'speechiness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'duration_ms',
 'duration_minutes']

The numeric features include audio measurements such as energy, danceability, loudness, tempo, duration, and other Spotify audio features.

### 4.5 Are there any artists that have more than 1 popular track?

I count how many times each artist appears, then keep artists with more than one track.

In [15]:
artist_counts = df["artist"].value_counts()
artists_with_more_than_one_track = artist_counts[artist_counts > 1]
artists_with_more_than_one_track

artist
Billie Eilish    3
Dua Lipa         3
Travis Scott     3
Justin Bieber    2
Harry Styles     2
Lewis Capaldi    2
Post Malone      2
Name: count, dtype: int64

Yes. Billie Eilish, Dua Lipa, and Travis Scott each have 3 tracks. Justin Bieber, Harry Styles, Lewis Capaldi, and Post Malone each have 2 tracks.

### 4.6 Who was the most popular artist?

In this dataset, I define the most popular artist as the artist with the highest number of tracks in the Top 50.

In [16]:
highest_artist_count = artist_counts.max()
most_popular_artists = artist_counts[artist_counts == highest_artist_count]
most_popular_artists

artist
Billie Eilish    3
Dua Lipa         3
Travis Scott     3
Name: count, dtype: int64

The most popular artists by track count are Billie Eilish, Dua Lipa, and Travis Scott. They are tied with 3 tracks each.

### 4.7 How many artists have their songs in the top 50 in total?

In [17]:
total_artists = df["artist"].nunique()
total_artists

40

There are 40 unique artists in the Top 50 dataset.

### 4.8 Are there any albums that have more than 1 popular track?

I count how many times each album appears, then keep albums with more than one track.

In [18]:
album_counts = df["album"].value_counts()
albums_with_more_than_one_track = album_counts[album_counts > 1]
albums_with_more_than_one_track

album
Future Nostalgia        3
Hollywood's Bleeding    2
Fine Line               2
Changes                 2
Name: count, dtype: int64

Yes. `Future Nostalgia` appears 3 times. `Hollywood's Bleeding`, `Fine Line`, and `Changes` each appear 2 times.

### 4.9 How many albums in total have their songs in the top 50?

In [19]:
total_albums = df["album"].nunique()
total_albums

45

There are 45 unique albums in the Top 50 dataset.

### 4.10 Which tracks have a danceability score above 0.7?

In [20]:
danceability_above_07 = df[df["danceability"] > 0.7][
    ["track_name", "artist", "danceability"]
].sort_values(by="danceability", ascending=False)

danceability_above_07

,track_name,artist,danceability
27,WAP (feat. Megan Thee Stallion),Cardi B,0.935
2,The Box,Roddy Ricch,0.896
39,Ride It,Regard,0.880
28,Sunday Best,Surfaces,0.878
33,Supalonely (feat. Gus Dapperton),BENEE,0.862
40,goosebumps,Travis Scott,0.841
49,SICKO MODE,Travis Scott,0.834
15,Toosie Slide,Drake,0.830
1,Dance Monkey,Tones And I,0.825
29,Godzilla (feat. Juice WRLD),Eminem,0.808


These tracks have danceability above 0.7, meaning they are among the more danceable songs in this dataset.

### 4.11 Which tracks have a danceability score below 0.4?

In [21]:
danceability_below_04 = df[df["danceability"] < 0.4][
    ["track_name", "artist", "danceability"]
].sort_values(by="danceability", ascending=True)

danceability_below_04

,track_name,artist,danceability
44,lovely (with Khalid),Billie Eilish,0.351


Only `lovely (with Khalid)` by Billie Eilish has danceability below 0.4.

### 4.12 Which tracks have their loudness above -5?

Loudness values are negative. Values closer to 0 are louder.

In [22]:
loudness_above_minus_5 = df[df["loudness"] > -5][
    ["track_name", "artist", "loudness"]
].sort_values(by="loudness", ascending=False)

loudness_above_minus_5

,track_name,artist,loudness
10,Tusa,KAROL G,-3.280
40,goosebumps,Travis Scott,-3.370
31,Break My Heart,Dua Lipa,-3.434
38,Hawái,Maluma,-3.454
12,Circles,Post Malone,-3.497
23,Mood (feat. iann dior),24kGoldn,-3.558
21,Adore You,Harry Styles,-3.675
49,SICKO MODE,Travis Scott,-3.714
48,Physical,Dua Lipa,-3.756
35,Rain On Me (with Ariana Grande),Lady Gaga,-3.764


These tracks have loudness above -5, so they are among the louder songs in this dataset.

### 4.13 Which tracks have their loudness below -8?

In [23]:
loudness_below_minus_8 = df[df["loudness"] < -8][
    ["track_name", "artist", "loudness"]
].sort_values(by="loudness", ascending=True)

loudness_below_minus_8

,track_name,artist,loudness
24,everything i wanted,Billie Eilish,-14.454
26,bad guy,Billie Eilish,-10.965
44,lovely (with Khalid),Billie Eilish,-10.109
47,If the World Was Ending - feat. Julia Michaels,JP Saxe,-10.086
15,Toosie Slide,Drake,-8.820
7,death bed (coffee for your head),Powfu,-8.765
36,HIGHEST IN THE ROOM,Travis Scott,-8.764
8,Falling,Trevor Daniel,-8.756
20,Savage Love (Laxed - Siren Beat),Jawsh 685,-8.520


These tracks are quieter compared with the rest of the dataset. Billie Eilish appears several times in this quieter group.

### 4.14 Which track is the longest?

In [24]:
longest_track = df[["track_name", "artist", "duration_minutes"]].sort_values(
    by="duration_minutes",
    ascending=False,
).head(1)

longest_track

,track_name,artist,duration_minutes
49,SICKO MODE,Travis Scott,5.213667


The longest track is `SICKO MODE` by Travis Scott, at about 5.21 minutes.

### 4.15 Which track is the shortest?

In [25]:
shortest_track = df[["track_name", "artist", "duration_minutes"]].sort_values(
    by="duration_minutes",
    ascending=True,
).head(1)

shortest_track

,track_name,artist,duration_minutes
23,Mood (feat. iann dior),24kGoldn,2.3421


The shortest track is `Mood (feat. iann dior)` by 24kGoldn, at about 2.34 minutes.

### 4.16 Which genre is the most popular?

I clean extra spaces from the genre names before counting them.

In [26]:
df["genre"] = df["genre"].str.strip()

genre_counts = df["genre"].value_counts()
most_popular_genre = genre_counts.head(1)
most_popular_genre

genre
Pop    14
Name: count, dtype: int64

Pop is the most popular genre, with 14 tracks.

### 4.17 Which genres have just one song in the top 50?

In [27]:
genres_with_one_song = genre_counts[genre_counts == 1]
genres_with_one_song

genre
Nu-disco                              1
R&B/Hip-Hop alternative               1
Pop/Soft Rock                         1
Pop rap                               1
Hip-Hop/Trap                          1
Dance-pop/Disco                       1
Disco-pop                             1
Dreampop/Hip-Hop/R&B                  1
Alternative/reggaeton/experimental    1
Chamber pop                           1
Name: count, dtype: int64

These genres each appear once in the Top 50. This means they are represented, but they are not common in this dataset.

### 4.18 How many genres in total are represented in the top 50?

In [28]:
total_genres = df["genre"].nunique()
total_genres

16

There are 16 genres represented in the Top 50 dataset.

### 4.19 Which features are strongly positively correlated?

Correlation shows whether two numeric features move together. A correlation close to 1 is strongly positive.

In [29]:
correlation_columns = [
    "energy",
    "danceability",
    "loudness",
    "acousticness",
    "speechiness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_minutes",
]

correlation_matrix = df[correlation_columns].corr()
correlation_pairs = []

for first_index, first_column in enumerate(correlation_columns):
    for second_column in correlation_columns[first_index + 1:]:
        correlation_pairs.append(
            {
                "feature_1": first_column,
                "feature_2": second_column,
                "correlation": correlation_matrix.loc[first_column, second_column],
            }
        )

correlation_pairs_df = pd.DataFrame(correlation_pairs)

strong_positive_correlations = correlation_pairs_df[
    correlation_pairs_df["correlation"] >= 0.7
].sort_values(by="correlation", ascending=False)

strong_positive_correlations

,feature_1,feature_2,correlation
1,energy,loudness,0.79164


Energy and loudness are strongly positively correlated. This means higher-energy songs also tend to be louder.

### 4.20 Which features are strongly negatively correlated?

A correlation close to -1 is strongly negative.

In [30]:
strong_negative_correlations = correlation_pairs_df[
    correlation_pairs_df["correlation"] <= -0.6
].sort_values(by="correlation", ascending=True)

strong_negative_correlations

,feature_1,feature_2,correlation
2,energy,acousticness,-0.682479


Energy and acousticness are strongly negatively correlated. This means more acoustic songs tend to have lower energy in this dataset.

### 4.21 Which features are not correlated?

Correlations close to 0 show little or no relationship.

In [31]:
not_correlated = correlation_pairs_df[
    correlation_pairs_df["correlation"].abs() < 0.1
].sort_values(by="correlation")

not_correlated.head(10)

,feature_1,feature_2,correlation
41,liveness,duration_minutes,-0.090188
35,instrumentalness,liveness,-0.087034
20,loudness,liveness,-0.069939
43,valence,duration_minutes,-0.039794
16,danceability,duration_minutes,-0.033763
39,liveness,valence,-0.033366
18,loudness,speechiness,-0.021693
12,danceability,instrumentalness,-0.017706
29,acousticness,duration_minutes,-0.010988
13,danceability,liveness,-0.006648


These feature pairs have correlations close to 0, so they do not show a strong relationship in this dataset. Correlation does not prove cause and effect; it only shows whether values tend to move together.

### 4.22 How does danceability compare between Pop, Hip-Hop/Rap, Dance/Electronic, and Alternative/Indie?

In [32]:
selected_genres = [
    "Pop",
    "Hip-Hop/Rap",
    "Dance/Electronic",
    "Alternative/Indie",
]

selected_genre_comparison = (
    df[df["genre"].isin(selected_genres)]
    .groupby("genre")[["danceability", "loudness", "acousticness"]]
    .mean()
    .loc[selected_genres]
)

selected_genre_comparison[["danceability"]].sort_values(
    by="danceability",
    ascending=False,
)

,danceability
genre,
Hip-Hop/Rap,0.765538
Dance/Electronic,0.755000
Pop,0.677571
Alternative/Indie,0.661750


Hip-Hop/Rap has the highest average danceability among these four genres. Alternative/Indie has the lowest average danceability.

### 4.23 How does loudness compare between Pop, Hip-Hop/Rap, Dance/Electronic, and Alternative/Indie?

In [33]:
selected_genre_comparison[["loudness"]].sort_values(
    by="loudness",
    ascending=False,
)

,loudness
genre,
Dance/Electronic,-5.338000
Alternative/Indie,-5.421000
Pop,-6.460357
Hip-Hop/Rap,-6.917846


Dance/Electronic has the loudest average loudness score because its value is closest to zero. Hip-Hop/Rap has the lowest average loudness among these four genres.

### 4.24 How does acousticness compare between Pop, Hip-Hop/Rap, Dance/Electronic, and Alternative/Indie?

In [34]:
selected_genre_comparison[["acousticness"]].sort_values(
    by="acousticness",
    ascending=False,
)

,acousticness
genre,
Alternative/Indie,0.583500
Pop,0.323843
Hip-Hop/Rap,0.188741
Dance/Electronic,0.099440


Alternative/Indie has the highest average acousticness. Dance/Electronic has the lowest average acousticness among these four genres.

## 5. Final Summary

This analysis explored the Spotify Top 50 Tracks of 2020 dataset.

Main findings:

- The dataset contains 50 tracks and 17 working features after cleaning.
- Billie Eilish, Dua Lipa, and Travis Scott are tied as the most popular artists by track count.
- Pop is the most common genre, followed by Hip-Hop/Rap.
- `WAP (feat. Megan Thee Stallion)` has the highest danceability score.
- `SICKO MODE` is the longest track, and `Mood (feat. iann dior)` is the shortest track.
- Energy and loudness are strongly positively correlated.
- Energy and acousticness are strongly negatively correlated.

## 6. Suggestions for Improvement

This analysis could be improved in several ways:

- Use a larger dataset with more songs and more years, so the findings are not based only on 50 tracks from 2020.
- Compare 2020 with other years to see whether music trends changed over time.
- Add charts to make artist, genre, and correlation patterns easier to understand visually.
- Group similar genres together, because the dataset has many small genre labels that appear only once.
- Include Spotify popularity scores or streaming counts if available, so audio features can be compared with actual popularity levels.

These improvements would make the analysis stronger and more useful for business decisions.

## 7. Limitations

This dataset only contains 50 songs from one year. The findings describe this specific dataset only and should not be treated as rules for all Spotify music.